#1. What is a tokenizer?

## A tokenizer is a component that converts text into a sequence of numerical token IDs that a language model can process. Language models do not directly understand raw text. They operate on numbers, usually token IDs that map to entries in the model's vocabulary.


---

```
Raw text

I am learning machine learning.

Tokenization

Tokens

["I", " am", " learning", " machine", " learning", "."]

Token-to-ID conversion

Token IDs

[40, 1097, ..., ...]


Model input

Numerical tensors
```
A token can represent:
- A complete word.
- Part of a word.
- Punctuation.
- Whitespace combined with a word.
- A special control token.
Important: A token is not necessarily the same thing as a word. For example, an uncommon word might be split into multiple subword tokens.







---



---



# 2. How tokenization works


## Tokenizers commonly use subword tokenization. Instead of requiring every possible word to exist in the vocabulary, they can split words into smaller pieces.

Three important approaches you'll encounter:
```
Algorithm Basic                                       idea	                                                 Common association
```
```
BPE (Byte Pair Encoding)	  Repeatedly merges frequent character or subword pairs                         GPT-style tokenizers

WordPiece	                  Builds subword units based on how useful they are for the vocabulary	        BERT

Unigram	                      selects a useful vocabulary	                                          SentencePiece-based models
```
These are algorithm families, not strict rules for every model. Different model families may use different implementations or variations.

Why subword tokenization?

Imagine the tokenizer encounters an unfamiliar word. Rather than failing because the whole word is missing, it may split it into smaller pieces that are already in its vocabulary.

This helps models handle:
- Uncommon words.
- Names and technical terminology.
- Different word forms.
- Multiple languages, depending on the tokenizer's training data.
Key terms:
- Vocabulary: The collection of tokens a tokenizer knows.
- Token ID: The integer assigned to a token.
- Encoding: Converting text into token IDs.
- Decoding: Converting token IDs back into readable text.
- Subword: A piece of a word rather than necessarily a complete word.



---



---



# 3. Load a tokenizer using AutoTokenizer
## Hugging Face provides AutoTokenizer, which automatically loads the correct tokenizer implementation for a model repository.

We generally use:

In [3]:
!pip install -q -U transformers tokenizers huggingface_hub

from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=False
)

print("Tokenizer loaded successfully!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully!


Explanation:
- AutoTokenizer selects the tokenizer implementation based on the model's configuration.
- from_pretrained() downloads or loads the tokenizer files from the Hugging Face Hub.
- model_id identifies the model repository.
- trust_remote_code=False avoids executing custom repository code. Some models require custom code, but don't enable it without reviewing the repository and understanding the implications.
The downloaded files are cached in the environment, so subsequent loads may use the local cache.

Inspect the tokenizer

In [4]:
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("Model maximum length:", tokenizer.model_max_length)

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)

print("BOS token:", tokenizer.bos_token)
print("BOS token ID:", tokenizer.bos_token_id)

print("UNK token:", tokenizer.unk_token)
print("UNK token ID:", tokenizer.unk_token_id)

Tokenizer class: Qwen2Tokenizer
Vocabulary size: 151643
Model maximum length: 131072
PAD token: <|endoftext|>
PAD token ID: 151643
EOS token: <|im_end|>
EOS token ID: 151645
BOS token: None
BOS token ID: None
UNK token: None
UNK token ID: None


Explanation:
Attribute	Meaning
```
vocab_size	                       Number of entries in the tokenizer's base vocabulary
model_max_length	               Tokenizer's configured maximum input length; check the model documentation for the actual supported context length
pad_token	                       Token used to fill shorter sequences in a batch
eos_token	                       End-of-sequence token
bos_token	                       Beginning-of-sequence token, if defined
unk_token	                       Unknown-token marker, if defined
```
Not every tokenizer defines all these special tokens. Some models use different conventions.

One important distinction: vocab_size is not necessarily the same as the number of embeddings in a model. A model may have additional embedding entries for added tokens.



---



---



# 4. Convert text into tokens and token IDs

Tokenization usually involves two related operations:
1. Split the input text into tokens.
2. Map each token to its corresponding integer ID.
Hugging Face provides several useful methods:
- tokenizer.tokenize(text) — returns token strings.
- tokenizer.convert_tokens_to_ids(tokens) — converts token strings to IDs.
- tokenizer.encode(text) — returns token IDs.
- tokenizer.decode(ids) — converts IDs back into text.

In [9]:
text = "I am learning finetuning"

tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

ids = tokenizer.convert_tokens_to_ids(tokens)
print("Original text:", text)
print("Tokens:", tokens)
print("Number of tokens:", len(tokens))

Tokens: ['I', 'Ġam', 'Ġlearning', 'Ġfin', 'et', 'uning']
Original text: I am learning finetuning
Tokens: ['I', 'Ġam', 'Ġlearning', 'Ġfin', 'et', 'uning']
Number of tokens: 6


## Conver tokens into ID's
Each token has a corresponding ID in the vocabulary. These IDs are what the model ultimately uses as input.

In [10]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

print("\nNumber of token IDs:", len(token_ids))

Tokens:
['I', 'Ġam', 'Ġlearning', 'Ġfin', 'et', 'uning']

Token IDs:
[40, 1079, 6832, 1875, 295, 37202]

Number of token IDs: 6


## Use encode() and decode()
Here, add_special_tokens=False tells the tokenizer not to add its configured special tokens automatically.

The exact default behavior can vary by tokenizer. Make it explicit when you need predictable preprocessing.

Important distinction:

tokenizer.tokenize() returns token strings, while tokenizer.encode() returns IDs. You will generally use tokenizer() or encode() when preparing actual model inputs.

In [11]:
encode_text = tokenizer.encode(
    text,
    add_special_tokens=False
)

print("Encode text:", encode_text)

decode_text = tokenizer.decode(
    encode_text,
    skip_special_tokens=True
)

print("Decode text:", decode_text)


Encode text: [40, 1079, 6832, 1875, 295, 37202]
Decode text: I am learning finetuning




---



---



# 5. Understand special tokens

Special tokens are reserved tokens that communicate structure or control information to a model.

Common examples include:

Token	Purpose
```
BOS	Marks the beginning of a sequence
EOS	Marks the end of a sequence
PAD	Fills unused positions in a batch
UNK	Represents an unknown token
SEP	Separates segments in certain model architectures
CLS	Represents a classification-related sequence position in models such as BERT
```
These are conceptual names. The actual token strings vary between model families.

For example, an instruction-tuned chat model may also have special tokens for:
- Beginning a user message.
- Beginning an assistant response.
- Separating message headers from message content.
- Ending a message or ending an assistant turn.
- Indicating a tool call.

In [12]:
print("All special tokens:")
print(tokenizer.all_special_tokens)

print("\nAll special token IDs:")
print(tokenizer.all_special_ids)

print("\nSpecial token map:")
print(tokenizer.special_tokens_map)

All special tokens:
['<|im_end|>', '<|endoftext|>', '<|im_start|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']

All special token IDs:
[151645, 151643, 151644, 151646, 151647, 151648, 151649, 151650, 151651, 151652, 151653, 151654, 151655, 151656]

Special token map:
{'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}




---



---



# 6. The tokenizer() method and model inputs
In real machine learning workflows, we generally use:
```
encoded = tokenizer(text)
```

This returns a BatchEncoding object containing information such as:
- input_ids: Numerical representation of the input.
- attention_mask: Indicates which positions should be attended to and which are padding.
Depending on the tokenizer and arguments, it can also return token type IDs and other information.

return_tensors="pt" requests PyTorch tensors instead of ordinary Python lists.

In [13]:
text = "Fine-tuning helps adapt a pretrained model to a specific task."

encoded = tokenizer(
    text,
    return_tensors="pt"
)

print(encoded)

{'input_ids': tensor([[63716,  2385, 37202,  8609, 10515,   264, 80669,  1614,   311,   264,
          3151,  3383,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [14]:
print("Input IDs:")
print(encoded["input_ids"])

print("\nAttention Mask:")
print(encoded["attention_mask"])

print("\nInput shape:")
print(encoded["input_ids"].shape)

print("\nAttention mask shape:")
print(encoded["attention_mask"].shape)

Input IDs:
tensor([[63716,  2385, 37202,  8609, 10515,   264, 80669,  1614,   311,   264,
          3151,  3383,    13]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Input shape:
torch.Size([1, 13])

Attention mask shape:
torch.Size([1, 13])




---



---



# 7. Padding and truncation
When processing multiple text sequences together, their lengths may differ.

For example:
```
Sentence 1: I am learning Python.
Sentence 2: I am learning Python and machine learning.
Sentence 3: Transformers are powerful.
```

Neural networks process batches using tensors. To create a rectangular tensor, sequences in a batch usually need the same length.

Two operations solve this:

Padding: Adds padding tokens to shorter sequences.

Truncation: Removes tokens from sequences that exceed a specified maximum length.

Common tokenizer arguments:
```
Argument	Purpose
padding=True	        Pads sequences to the longest sequence in the current batch
padding="max_length"	Pads every sequence to the specified maximum length
truncation=True	        Truncates sequences exceeding the specified length
max_length=...	        Sets the target maximum sequence length
return_tensors="pt"	    Returns PyTorch tensors
```

In [15]:
texts = [
    "I am learning Python.",
    "I am learning machine learning and deep learning.",
    "Transformers are useful for building language models."
]

batch = tokenizer(
    texts,
    padding=True,
    return_tensors="pt"
)

print("Input IDs:")
print(batch["input_ids"])

print("\nAttention Masks:")
print(batch["attention_mask"])

print("\nBatch shape:")
print(batch["input_ids"].shape)

Input IDs:
tensor([[    40,   1079,   6832,  13027,     13, 151643, 151643, 151643, 151643],
        [    40,   1079,   6832,   5662,   6832,    323,   5538,   6832,     13],
        [  8963,    388,    525,   5390,    369,   4752,   4128,   4119,     13]])

Attention Masks:
tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])

Batch shape:
torch.Size([3, 9])


Explanation:

padding=True pads each sequence to the length of the longest sequence in this batch.

The padding token ID depends on the tokenizer. The tokenizer handles that automatically when a suitable padding token is configured.

In [16]:
batch = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=32,
    return_tensors="pt"
)

print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)

Input IDs shape: torch.Size([3, 32])
Attention mask shape: torch.Size([3, 32])


**Important: Truncation can remove useful information. Choose a maximum length that makes sense for your task and model. Also, the model's supported context length is an architectural constraint; don't assume a tokenizer's configured maximum automatically guarantees that every model can accept that many tokens.**



---



---



# 8. Attention masks — why they matter
An attention mask tells the model which positions contain actual input tokens and which positions are padding.

Typically:
- 1 = actual token; include it in attention.
- 0 = padding position; mask it out.
Example:
```
Input IDs:       [101, 205, 309, 0, 0]
Attention Mask:  [  1,   1,   1, 0, 0]
```

In [17]:
for i in range(len(texts)):
    print(f"\nSequence {i + 1}")

    print("Input IDs:")
    print(batch["input_ids"][i].tolist())

    print("Attention Mask:")
    print(batch["attention_mask"][i].tolist())


Sequence 1
Input IDs:
[40, 1079, 6832, 13027, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643]
Attention Mask:
[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Sequence 2
Input IDs:
[40, 1079, 6832, 5662, 6832, 323, 5538, 6832, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643]
Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Sequence 3
Input IDs:
[8963, 388, 525, 5390, 369, 4752, 4128, 4119, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 1

**The mask allows the model to distinguish real tokens from padding.
Without correct masking, padding positions could interfere with attention computations. Most Hugging Face model implementations handle the mask when you pass the tokenizer's complete output to the model.
A useful detail: attention masks are not only about padding. Other model architectures and tasks may use masks for different purposes, too.**




---


---



# 9. Chat templates — essential for instruction-tuned LLMs

Many modern LLMs are trained to process conversations, not just plain text.

A conversation can contain multiple messages with different roles:
- system: Defines the assistant's behavior or instructions.
- user: Contains the user's message.
- assistant: Contains the assistant's response.
A chat template converts these messages into the exact format expected by the model.

Different model families use different formats. For example, one model might use explicit role markers, while another uses a different message structure.

Why does this matter?

If you fine-tune an instruction model using the wrong conversation format, the model may not learn the behavior you intend. The template is part of the model's input format.

In [18]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [19]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful AI tutor."
    },
    {
        "role": "user",
        "content": "Explain tokenization in simple terms."
    },
    {
        "role": "assistant",
        "content": "Tokenization converts text into smaller units called tokens."
    }
]

formatted_conversation = tokenizer.apply_chat_template(
    messages,
    tokenize=False
)

print(formatted_conversation)

<|im_start|>system
You are a helpful AI tutor.<|im_end|>
<|im_start|>user
Explain tokenization in simple terms.<|im_end|>
<|im_start|>assistant
Tokenization converts text into smaller units called tokens.<|im_end|>



# 10. Token count and sequence length

Token count is important when working with LLMs because models have context-length limits, and API or inference costs are often based on tokens rather than words.

A sentence with 100 words does not necessarily contain 100 tokens.

You can calculate the number of tokens using:
```
len(tokenizer.encode(text))
```

In [20]:
sentences = [
    "Hello, how are you?",
    "I am learning Hugging Face tokenizers.",
    "Large language models process sequences of tokens."
]

for sentence in sentences:
    ids = tokenizer.encode(sentence)

    print("Text:", sentence)
    print("Token count:", len(ids))
    print("-" * 50)

Text: Hello, how are you?
Token count: 6
--------------------------------------------------
Text: I am learning Hugging Face tokenizers.
Token count: 9
--------------------------------------------------
Text: Large language models process sequences of tokens.
Token count: 8
--------------------------------------------------


This counts the IDs produced by encode(), including any special tokens added by default.

If you want to count only the text tokens:

In [21]:
ids = tokenizer.encode(
    text,
    add_special_tokens=False
)

print("Text token count:", len(ids))

Text token count: 13




---



---



# 11. Save and reload a tokenizer
After configuring or modifying a tokenizer, you can save it locally using save_pretrained().

You can then reload it using from_pretrained().

This is useful for:
- Reusing the same tokenizer across training and inference.
- Preserving added tokens.
- Maintaining consistency between training and deployment.
- Saving tokenizer files alongside fine-tuned model weights.

In [22]:
save_path = "/content/tokenizer_test"

tokenizer.save_pretrained(save_path)

print("Tokenizer saved successfully!")

Tokenizer saved successfully!


In [23]:
reloaded_tokenizer = AutoTokenizer.from_pretrained(
    save_path
)

test_text = "I am learning LLM engineering."

original_ids = tokenizer.encode(test_text)
reloaded_ids = reloaded_tokenizer.encode(test_text)

print("Original IDs:", original_ids)
print("Reloaded IDs:", reloaded_ids)

print("Identical:", original_ids == reloaded_ids)

Original IDs: [40, 1079, 6832, 444, 10994, 14667, 13]
Reloaded IDs: [40, 1079, 6832, 444, 10994, 14667, 13]
Identical: True




---



---



# 12. Add custom tokens — an introduction
Sometimes you need to introduce new tokens into a tokenizer.

For example, you might want to add a domain-specific marker such as:
```
<DISEASE>
```
Hugging Face supports adding tokens using add_tokens() and special tokens using add_special_tokens().

In [24]:
from transformers import AddedToken

new_token = AddedToken(
    "<DOMAIN>",
    special=True
)

number_added = tokenizer.add_special_tokens({
    "additional_special_tokens": [new_token]
})

print("New tokens added:", number_added)
print("Updated vocabulary size:", len(tokenizer))

New tokens added: 1
Updated vocabulary size: 151666


## Explanation:
- add_special_tokens() adds the token to the tokenizer's special-token configuration.
- number_added tells you how many new vocabulary entries were added.
- len(tokenizer) reports the tokenizer's total vocabulary size, including added tokens.
Important for later model fine-tuning:

When you add new tokens to a tokenizer, the model may also need its embedding matrix resized:

In [ ]:
model.resize_token_embeddings(len(tokenizer))